<a href="https://colab.research.google.com/github/kader-xai/data-science-roadmap/blob/main/module_69_feature_stores_feast.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

> 🧊 **Module 69 — Feature Stores (Feast)** &nbsp;·&nbsp; part of [`data-science-roadmap`](https://github.com/kader-xai/data-science-roadmap)

> Phase 9 · Production Gaps


# Module 69 — Feature Stores

> A model trained on `user_avg_purchase_last_30d` will silently break the day production serves it a *slightly different* version of that same feature — re-implemented in a different language, computed over a slightly different window, missing the last-hour update. This is **training-serving skew**, the most expensive bug in classical ML. **Feature stores** solve it: define a feature **once**, serve the same value to training and online inference.
>
> This module walks the canonical open-source one — **Feast** — and shows when you actually need a feature store (often you don't).

### What you'll cover
1. The training-serving skew problem (and why it's invisible)
2. What a feature store is: **offline + online + registry**
3. Feast's three-layer architecture
4. `Entity`, `FeatureView`, `Field` — the core abstractions
5. **Point-in-time correctness** — the unsexy core feature
6. `feast apply` → `materialize` → online serving
7. Hosted feature stores: **Tecton, Hopsworks, Vertex AI FS, Databricks FS, SageMaker FS**
8. **Streaming features** — events to the online store
9. Feature stores for **LLMs / RAG** — embeddings, last-N session signals
10. The "do you actually need one?" decision tree


## 1 · The training-serving skew problem

Imagine a fraud-detection model that uses these features:

| Feature | Training-time source | Production source |
|---|---|---|
| `user_avg_purchase_last_30d` | Spark job on the data warehouse | Online API computing on the last 30 days of Postgres |
| `user_country` | Snowflake `users` table | Microservice that returns the *current* country |
| `is_weekend` | derived from `event_ts` in batch | derived from `now()` at inference |
| `embedding_v2` | offline batch from `sentence-transformers` | live embedding service |

Each row trains the model on features computed **one way**. Production serves features computed **another way**:

- Window boundaries differ by a few seconds.
- "Current country" at inference != "country at the time of the historical event."
- Embedding model version drifts on production but training data has the *old* version.
- A nightly batch lags 6 hours; online is real-time.

The model **silently** does worse in production. Offline AUC = 0.91, online conversion lift = 0.0. **No exception ever fires.** That's training-serving skew.

## 2 · The three layers of a feature store

```
   ┌─────────────────────── REGISTRY ───────────────────────┐
   │  Single source of truth for feature definitions        │
   │  - feature view names, schemas, owners, descriptions   │
   │  - lineage (which Snowflake table feeds which feature) │
   └────────────────────────────────────────────────────────┘
                      ▲                ▲
                      │ definitions    │ definitions
                      │                │
   ┌────────── OFFLINE STORE ─────┐  ┌── ONLINE STORE ──┐
   │  BigQuery / Snowflake /       │  │  Redis / DynamoDB │
   │  Redshift / Parquet on S3     │  │  Bigtable / PG    │
   │                                │  │                  │
   │  for TRAINING                  │  │  for SERVING     │
   │  - massive scans              │  │  - ~1 ms lookups │
   │  - point-in-time joins        │  │  - keyed by entity│
   └───────────────────────────────┘  └───────────────────┘
                      ▲                        ▲
                      │ batch / stream         │ same definitions, same code
                 ┌──────────────────┐         │
                 │  RAW SOURCES     │ ────────┘ (materialization)
                 │  warehouse,      │
                 │  Kafka, files    │
                 └──────────────────┘
```

A feature store keeps **one definition per feature**, computes it twice — once into the offline store for training, once into the online store for inference — and **guarantees they're the same logic**.

## 3 · Feast setup

In [ ]:
!pip -q install "feast[redis]" pandas pyarrow

In [ ]:
# Initialise a fresh Feast repo
import subprocess, os
os.makedirs("/content/feast_demo", exist_ok=True)
res = subprocess.run(
    ["feast","init","fraud","--minimal"],
    cwd="/content/feast_demo", capture_output=True, text=True,
)
print(res.stdout)
print(res.stderr)

`feast init` creates:
```
fraud/
  feature_store.yaml      # registry/online/offline backend config
  example_repo.py         # your feature definitions
  data/                    # parquet files for the demo
```

For real projects, **`feature_store.yaml`** points the offline store at Snowflake / BigQuery / Redshift / S3-Parquet, and the online store at Redis / DynamoDB / Bigtable / Postgres.

## 4 · Entities, FeatureViews, Fields

In [ ]:
defs_py = '''
# fraud/repo.py — the feature definitions
from datetime import timedelta
from feast import Entity, Feature, FeatureView, Field, FileSource, ValueType
from feast.types import Float32, Int64, String

# 1. ENTITY — the thing features describe (a user, a merchant, an item).
user = Entity(name="user_id", value_type=ValueType.INT64,
              description="Unique user id")

# 2. SOURCE — where the raw, time-stamped data comes from
purchases_source = FileSource(
    path="data/purchases.parquet",
    timestamp_field="event_ts",     # the event time — critical for point-in-time correctness
    created_timestamp_column="created_ts",   # when the row landed (optional)
)

# 3. FEATURE VIEW — group of features that share an entity + source
user_purchases_30d = FeatureView(
    name="user_purchases_30d",
    entities=[user],
    ttl=timedelta(days=7),        # how long online-store values stay fresh
    schema=[
        Field(name="avg_purchase_30d",  dtype=Float32),
        Field(name="count_purchase_30d", dtype=Int64),
        Field(name="last_country",       dtype=String),
    ],
    source=purchases_source,
    online=True,                  # push these into the online store
    tags={"team": "fraud", "owner": "kim@example.com"},
)
'''
print(defs_py)

**Three nouns you'll see in every feature store.**

| Concept | Definition | Example |
|---|---|---|
| **Entity** | the thing features describe | `user_id`, `merchant_id`, `device_id` |
| **Feature View** | a group of features that share an entity + a source | `user_purchases_30d` |
| **Field** | one column with a typed dtype | `avg_purchase_30d: Float32` |

Plus **`FeatureService`** (a named collection of FeatureViews the model uses, version-pinned).

## 5 · Point-in-time correctness — the unsexy core feature

Suppose at training time you want, for **row (user=42, event_ts=2024-08-15 10:00 UTC)**, the value of `avg_purchase_30d` **as known on that exact day**. The naive approach is dangerous:

```sql
-- BAD: leaks future data into training!
SELECT user_id, AVG(amount) AS avg_purchase_30d
FROM purchases
WHERE user_id = 42 AND event_ts > '2024-07-15'
GROUP BY user_id;
```

The above runs *today* and pulls in purchases that happened *after* the training row's event. The model now "knows the future" — it overfits to leaked signal during training, then collapses in production.

A feature store's **`get_historical_features`** call enforces point-in-time correctness with an `AS OF` join:

In [ ]:
hist_call = '''
import pandas as pd
from feast import FeatureStore

store = FeatureStore(repo_path="fraud")

# An "entity dataframe" — one row per training example
entity_df = pd.DataFrame({
    "user_id":   [42, 42, 18],
    "event_ts":  [
        pd.Timestamp("2024-08-15 10:00", tz="UTC"),
        pd.Timestamp("2024-09-01 18:30", tz="UTC"),
        pd.Timestamp("2024-08-30 12:00", tz="UTC"),
    ],
})

training_df = store.get_historical_features(
    entity_df=entity_df,
    features=[
        "user_purchases_30d:avg_purchase_30d",
        "user_purchases_30d:count_purchase_30d",
        "user_purchases_30d:last_country",
    ],
).to_df()
'''
print(hist_call)

**Internally**, Feast issues an `AS OF` join: for each `(user_id, event_ts)` row in your entity dataframe, it grabs the **latest feature value with `feature_event_ts <= entity_event_ts`**. No leakage. Period.

This is the single most important thing a feature store does. Without it, every team eventually invents an `AS OF` join in dbt or Spark and gets it slightly wrong.

## 6 · `feast apply` → `materialize` → online serving

```bash
# 1. Register your definitions (creates / updates the registry)
$ feast apply

# 2. Materialise feature values into the online store
$ feast materialize \
    --start-date 2024-08-01T00:00:00 \
    --end-date   2024-10-01T00:00:00

# (or "materialize-incremental" for daily updates)
$ feast materialize-incremental $(date -u +"%Y-%m-%dT%H:%M:%S")
```

After materialisation, you can fetch in <1 ms in production:

In [ ]:
online_call = '''
features = store.get_online_features(
    features=[
        "user_purchases_30d:avg_purchase_30d",
        "user_purchases_30d:count_purchase_30d",
        "user_purchases_30d:last_country",
    ],
    entity_rows=[{"user_id": 42}, {"user_id": 18}],
).to_dict()

# At inference time, this is ~1 ms — a Redis lookup keyed by user_id.
print(features)
'''
print(online_call)

**Same Python call shape** in training and serving. Same feature names. **Same logic.** Skew gone.

> 🔄 Most teams run **`feast materialize-incremental`** as an Airflow / Prefect / Dagster (M68) task every N minutes for fresh-enough features. Real-time streaming features (next section) bypass batch entirely.

## 7 · Hosted alternatives

| Tool | Notes |
|---|---|
| **Feast** (OSS) | the canonical OSS feature store; **you run the infra** |
| **Tecton** | enterprise commercial Feast cousin; same model + managed + streaming compute layer |
| **Hopsworks** | OSS + commercial; strong online store + model registry combo |
| **Vertex AI Feature Store** | Google Cloud native; BigQuery offline + Bigtable online |
| **Databricks Feature Store** | tightly coupled to Unity Catalog + MLflow |
| **SageMaker Feature Store** | AWS-native; S3 + DynamoDB |
| **Chronon** (Airbnb OSS) | streaming-first feature store; aggregations over Kafka |
| **MetricFlow** + **dbt Semantic Layer** | not a feature store; same problem at the metric layer |

**The 2025 pattern** at most teams: **Feast** as the canonical layer with **Snowflake/BigQuery offline + Redis online**, materialised on a schedule. If you're already paying Databricks / GCP / AWS, their built-in feature store removes the operational burden.

## 8 · Streaming features

Some features can't wait for a nightly batch — fraud features (`count_logins_last_5min`), real-time recsys (`items_viewed_in_session`), live bidding (`current_user_intent`). Two patterns:

### Push to online store
Your stream processor (Flink / Spark Streaming / Materialize / Kafka Streams) writes directly to the online store keyed by entity. Feast's "push API" then exposes the values:

```python
store.push(
    push_source_name="user_session_features",
    df=pd.DataFrame({"user_id":[42], "items_viewed_5min":[3], "event_ts":[now()]}),
    to=PushMode.ONLINE,
)
```

### On-demand transforms
Some features are cheap to compute *at request time* from raw inputs:

```python
@on_demand_feature_view(
    sources=[user_purchases_30d],
    schema=[Field(name="purchase_velocity", dtype=Float32)],
)
def purchase_velocity(features_df):
    return pd.DataFrame({"purchase_velocity": features_df["count_purchase_30d"] / 30.0})
```

These run **inside the Feast call** at inference time — no materialisation needed.

## 9 · Feature stores for LLM / RAG apps

Feature stores aren't only for tabular ML. For LLM apps in 2025 they take three shapes:

| Use | What goes in the store |
|---|---|
| **User profile features for personalised RAG** | `last_purchase_category`, `tier`, `language_preference`, `recent_topics` |
| **Session signals** | last N user messages, ongoing tool-call results, time-of-day |
| **Pre-computed embeddings** | document embeddings (overlap with vector DBs — M42) |
| **Tool/Agent metrics** | success rate of `web_search` for this user, fallback model usage |
| **Conversation memory** | summary embeddings keyed by `user_id` |

The line between **feature store** and **vector DB** (M42) is blurring. Engines like **Tecton + LanceDB**, **Fennel**, **Featureform** explicitly bridge both. A simple rule:
- **Vector DB** for similarity search on high-dim vectors.
- **Feature store** for low-cardinality, low-dim signals keyed by entity (~ a row per user).

Use both. They're complementary, not competing.

## 10 · Do you actually need a feature store?

**Most teams don't.** Set a feature store up too early and you ship slower without the skew problem to justify the complexity. The decision tree:

```
Do features come from > 1 system (warehouse + microservice + stream)?
   │
   ├─ No  → just compute in your training job; you don't need a feature store.
   │
   └─ Yes
       │
       Do you have > 1 model sharing the same features?
       │
       ├─ No → put feature SQL in dbt; load directly at training; live with the skew
       │      (~80% of "ML teams" are here in 2025).
       │
       └─ Yes → do you need point-in-time correctness?
                 │
                 ├─ No → Redis + a daily Airflow job is enough; the "feature store" is a folder
                 │       of SQL + a Redis cache layer.
                 │
                 └─ Yes → reach for a feature store.
```

### Signs you've outgrown ad-hoc feature engineering

- Two models compute "user activity last 7d" *slightly differently* and the offline / online numbers don't match.
- A nightly Airflow DAG produces features; the online API recomputes them; **someone has to keep them in sync**.
- You have a "Wiki page of features" — and it's wrong.
- Your training job re-runs the same feature SQL **for every experiment**.

### Anti-patterns
- ❌ **One giant `everything` Feature Service** — keep services scoped per model.
- ❌ **No SLA on materialisation** — the online store is silently stale and predictions drift.
- ❌ **Online and offline schemas drift** — enforce schemas via the registry; reject mismatched columns at `apply` time.
- ❌ **Per-feature TTL guesswork** — match TTL to your materialisation cadence.

## ✅ Recap
- **Training-serving skew** is the most expensive ML bug. Feature stores fix it.
- A feature store has three layers: **registry**, **offline store**, **online store**.
- **Feast** (OSS) is the canonical tool; hosted alternatives exist for every cloud.
- The killer feature is **point-in-time correctness** via `AS OF` joins.
- Most teams **don't** need one. Reach for it when ≥ 2 models share features computed across ≥ 2 systems.
- Pair feature stores with vector DBs (M42) for LLM apps with both signal and similarity needs.

Next: **M70 — LLM Evals**.
